In [1]:
import pandas as pd

train_df = pd.read_csv("challenge2_train.csv")
print("Training data shape:", train_df.shape)
train_df.head()

Training data shape: (9247, 19)


,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


In [2]:
print("Total missing values:", train_df.isna().sum().sum())
print("Duplicate IDs:", train_df["ID"].duplicated().sum())

revenue_summary = train_df["Revenue"].value_counts(dropna=False).to_frame("Count")
revenue_summary["Percent"] = (
    revenue_summary["Count"] / len(train_df) * 100
).round(2)

display(revenue_summary)

Total missing values: 0
Duplicate IDs: 0


,Count,Percent
Revenue,,
False,7816,84.52
True,1431,15.48


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = train_df.drop(columns=["ID", "Revenue"])
y = train_df["Revenue"]

categorical_features = [
    "Month", "OperatingSystems", "Browser", "Region",
    "TrafficType", "VisitorType", "Weekend",
]
numeric_features = X.columns.difference(
    categorical_features, sort=False
).tolist()

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        categorical_features,
    ),
])

print("Feature matrix:", X.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Feature matrix: (9247, 17)
Numeric features: 10
Categorical features: 7


In [4]:
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import StratifiedGroupKFold

# Keep sessions with identical features in the same fold.
groups = pd.util.hash_pandas_object(X, index=False)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv = list(splitter.split(X, y, groups=groups))

f1_positive = make_scorer(f1_score, pos_label=True, zero_division=0)

fold_summary = pd.DataFrame([
    {
        "Fold": fold,
        "Validation rows": len(val_idx),
        "Purchase rate (%)": round(100 * y.iloc[val_idx].mean(), 2),
    }
    for fold, (_, val_idx) in enumerate(cv, start=1)
])

display(fold_summary)

,Fold,Validation rows,Purchase rate (%)
0,1,1850,15.46
1,2,1849,15.47
2,3,1850,15.51
3,4,1849,15.47
4,5,1849,15.47


### Preprocessing and Validation

The training dataset contains 9,247 sessions with no missing values.
Purchases account for 15.48% of sessions. We exclude `ID` from the
predictors and use `Revenue` as the target.

We standardize numeric features and one-hot encode categorical features.
Preprocessing is fitted inside each model pipeline on the training
portion of each fold.

We use five stratified group folds to keep sessions with identical
predictor values in the same fold. All models use the same splits and
are compared primarily using F1 for the positive class (`Revenue=True`).